# 02 — Feature Engineering
## Classical vs Multi-Agent · Reply × LUISS 2026

**Obiettivo:** Aggregare i dataset puliti per **ROTTA** (aeroporto_partenza → aeroporto_arrivo),
costruire feature significative e produrre `features_classical.csv` pronto per la detection.

---

### Strategia di aggregazione

| Dataset | Granularità raw | Unità aggregazione |
|---------|----------------|--------------------|
| `allarmi_clean.csv` | 1 riga = 1 occorrenza di 1 tipo di metrica per rotta/data | **ROTTA** |
| `viaggiatori_clean.csv` | 1 riga = 1 record viaggiatore per rotta/data | **ROTTA** |

> **Perché non si fa un join riga-per-riga?**  
> ALLARMI è una tabella pivot con ~30 tipi di OCCORRENZE per rotta/data → granularità incompatibile
> con VIAGGIATORI. Il join corretto avviene a livello di aggregazione: prima aggrego ciascun dataset
> separatamente, poi faccio un **outer join** sulle 567 rotte distinte.

---
**Output:**
- `data/processed/features_classical.csv` — 567 rotte × 63 feature
- `data/processed/feature_cols.json` — lista feature numeriche per i modelli ML

In [71]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# Percorsi — funziona sia da notebooks/ che dalla root
ROOT     = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"

print(f"ROOT:     {ROOT}")
print(f"PROC_DIR: {PROC_DIR}")

ROOT:     /Users/danielegiovanardi/classical-vs-multiagent/classical-vs-multiagent
PROC_DIR: /Users/danielegiovanardi/classical-vs-multiagent/classical-vs-multiagent/data/processed


## 1. Caricamento dati puliti

In [72]:
allarmi     = pd.read_csv(PROC_DIR / "allarmi_clean.csv")
viaggiatori = pd.read_csv(PROC_DIR / "viaggiatori_clean.csv")

print(f"ALLARMI clean:     {allarmi.shape[0]:>5} righe × {allarmi.shape[1]:>2} col")
print(f"VIAGGIATORI clean: {viaggiatori.shape[0]:>5} righe × {viaggiatori.shape[1]:>2} col")

ALLARMI clean:      5080 righe × 22 col
VIAGGIATORI clean:  5095 righe × 33 col


## 2. Costruzione chiave ROTTA

La chiave è `AREOPORTO_PARTENZA-AREOPORTO_ARRIVO` (codici IATA uppercase, già normalizzati in preprocessing).

In [73]:
allarmi["ROTTA"]     = (allarmi["AREOPORTO_PARTENZA"].str.upper()
                        + "-" + allarmi["AREOPORTO_ARRIVO"].str.upper())
viaggiatori["ROTTA"] = (viaggiatori["AREOPORTO_PARTENZA"].str.upper()
                        + "-" + viaggiatori["AREOPORTO_ARRIVO"].str.upper())

rotte_a = set(allarmi["ROTTA"].unique())
rotte_v = set(viaggiatori["ROTTA"].unique())
comuni  = rotte_a & rotte_v

print(f"Rotte distinte in ALLARMI:     {len(rotte_a)}")
print(f"Rotte distinte in VIAGGIATORI: {len(rotte_v)}")
print(f"Rotte in comune:               {len(comuni)}")
print(f"Solo in ALLARMI:               {len(rotte_a - rotte_v)}")
print(f"Solo in VIAGGIATORI:           {len(rotte_v - rotte_a)}")

Rotte distinte in ALLARMI:     368
Rotte distinte in VIAGGIATORI: 467
Rotte in comune:               268
Solo in ALLARMI:               100
Solo in VIAGGIATORI:           199


## 3. Funzioni di supporto

In [74]:
def safe_mode(x):
    """Mode di una serie pandas; restituisce 'ND' se la serie è vuota."""
    m = x.mode()
    return m.iloc[0] if len(m) > 0 else "ND"

def safe_div(a, b):
    """Divisione sicura vettorizzata: restituisce 0.0 dove b == 0."""
    return np.where(b > 0, a / b, 0.0)

print("✓ Funzioni helper definite")

✓ Funzioni helper definite


## 4. Aggregazione ALLARMI per ROTTA

### 4.1 Pivot OCCORRENZE → colonne numeriche

ALLARMI ha ~30 tipi di OCCORRENZE per rotta (es. "Allarmi generati da INTERPOL", "Voli con Allarmi", ecc.).
Facciamo un **pivot_table**: ogni tipo di OCCORRENZA diventa una colonna numerica (somma del TOT) per ROTTA.

In [75]:
# Pivot: una colonna per tipo di OCCORRENZA, valore = somma TOT per ROTTA
occ_pivot = allarmi.pivot_table(
    index      = "ROTTA",
    columns    = "OCCORRENZE",
    values     = "TOT",
    aggfunc    = "sum",
    fill_value = 0
).reset_index()
occ_pivot.columns.name = None

# Rinomina in snake_case leggibile
RENAME_OCC = {
    "Viaggiatori entrati nel Sistema"             : "vg_entrati_occ",
    "Viaggiatori con Allarmi"                     : "vg_con_allarmi",
    "Viaggiatori investigati"                     : "vg_investigati_occ",
    "Voli con Allarmi"                            : "voli_con_allarmi",
    "Voli disponibili in ingresso al Sistema"     : "voli_disponibili",
    "Voli investigati (SDI/NSIS - INTERPOL - TSC)": "voli_investigati",
    "Voli solo visualizzati, ma NON investigati"  : "voli_non_investigati",
    "Allarmi generati"                            : "allarmi_generati",
    "Allarmi generati da SDI/NSIS"                : "allarmi_sdi_occ",
    "Allarmi generati da INTERPOL"                : "allarmi_interpol_occ",
    "Allarmi generati da BCS"                     : "allarmi_bcs_occ",
    "Allarmi Chiusi"                              : "allarmi_chiusi",
    "Allarmi Chiusi con Azione (CC.xx)"           : "allarmi_chiusi_azione",
    "Allarmi NON Chiusi"                          : "allarmi_non_chiusi",
    "Allarmi Rilevanti"                           : "allarmi_rilevanti",
    "Respinto/a"                                  : "vg_respinti_occ",
    "Errata segnalazione SDI"                     : "err_sdi",
    "Errata segnalazione NSIS"                    : "err_nsis",
    "Errata segnalazione BCS"                     : "err_bcs",
    "Nulla a procedere SDI"                       : "np_sdi",
    "Nulla a procedere NSIS"                      : "np_nsis",
    "Nulla a procedere INT"                       : "np_int",
    "Notifica Atti/Provv"                         : "notifica_atti",
    "Mancato aggiornamento SDI"                   : "mancato_agg_sdi",
    "Mancato aggiornamento Schengen NSIS"         : "mancato_agg_nsis",
    "Inammissibilita Schengen - Art.24"           : "inammissib_schengen",
    "ALLARMATI"                                   : "allarmati_occ",
    "Altro"                                       : "altro_occ",
    "N/C"                                         : "nc_occ",
    "???"                                         : "unknown_occ",
}
occ_pivot = occ_pivot.rename(columns=RENAME_OCC)

print(f"occ_pivot: {occ_pivot.shape[0]} rotte × {occ_pivot.shape[1]} colonne")

occ_pivot: 368 rotte × 31 colonne


### 4.2 Feature % motivo allarme da MOTIVO_ALLARME

`pct_interpol`, `pct_sdi`, ecc. si calcolano da `MOTIVO_ALLARME` (743 record INTERPOL)  
anziché da OCCORRENZE="Allarmi generati da INTERPOL" (solo 8 record) — dati molto più ricchi.

In [76]:
# Conteggi per motivo per rotta
motivo_counts = (
    allarmi.groupby("ROTTA")["MOTIVO_ALLARME"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)
motivo_counts.columns.name = None

# Percentuali normalizzate [0,1]
motivo_totale = motivo_counts.drop(columns=["ROTTA"]).sum(axis=1)
for col in ["INTERPOL", "SDI", "NSIS", "TSC", "Manuale"]:
    if col in motivo_counts.columns:
        motivo_counts[f"pct_{col.lower()}"] = safe_div(
            motivo_counts[col], motivo_totale
        ).clip(0, 1)
        motivo_counts = motivo_counts.drop(columns=[col])

print(f"motivo_counts: {motivo_counts.shape}")
print("\nStatistiche pct_interpol:")
print(motivo_counts["pct_interpol"].describe().to_string())

motivo_counts: (351, 6)

Statistiche pct_interpol:
count   351.0000
mean      0.1857
std       0.2247
min       0.0000
25%       0.0000
50%       0.1471
75%       0.2500
max       1.0000


### 4.3 Metadati per rotta + log-transform di tot_allarmi

In [77]:
# Metadati: zona geografica, paese, volume totale allarmi
meta_allarmi = allarmi.groupby("ROTTA").agg(
    ZONA                   = ("ZONA",      "first"),
    PAESE_PART             = ("PAESE_PART", "first"),
    n_osservazioni_allarmi = ("TOT",        "count"),
    tot_allarmi_sum        = ("TOT",        "sum"),
).reset_index()

# ── FIX: Log-transform per tot_allarmi_sum (max=103254 → distribuzione skewed) ──
# log1p(x) = log(1+x) gestisce x=0 senza problemi
meta_allarmi["tot_allarmi_log"] = np.log1p(meta_allarmi["tot_allarmi_sum"])

print("tot_allarmi_sum (grezzo):")
print(meta_allarmi["tot_allarmi_sum"].describe().apply(lambda x: f"{x:,.0f}").to_string())
print("\ntot_allarmi_log (dopo log1p — max≈11.5):")
print(meta_allarmi["tot_allarmi_log"].describe().to_string())

tot_allarmi_sum (grezzo):
count        368
mean       1,139
std        7,547
min            0
25%           20
50%          259
75%          679
max      103,254

tot_allarmi_log (dopo log1p — max≈11.5):
count   368.0000
mean      4.8834
std       2.3430
min       0.0000
25%       3.0266
50%       5.5607
75%       6.5220
max      11.5450


### 4.4 Merge componenti ALLARMI + derived ratios

In [78]:
# Combina metadati + pivot occorrenze + percentuali motivo
agg_allarmi = (
    meta_allarmi
    .merge(occ_pivot,     on="ROTTA", how="left")
    .merge(motivo_counts, on="ROTTA", how="left")
)

# ── FIX: clip negativi dal pivot OCCORRENZE ──────────────────────────────────
# np_sdi e voli_non_investigati hanno TOT < 0 nel dataset sorgente.
# Clip a 0 su tutti i numerici prima di calcolare i ratio derivati.
_num_cols = agg_allarmi.select_dtypes(include="number").columns
agg_allarmi[_num_cols] = agg_allarmi[_num_cols].clip(lower=0)

# Tasso chiusura allarmi (chiusi / chiusi+non_chiusi)
agg_allarmi["tasso_chiusura"] = safe_div(
    agg_allarmi["allarmi_chiusi"],
    agg_allarmi["allarmi_chiusi"] + agg_allarmi["allarmi_non_chiusi"]
).clip(0, 1)

# Tasso rilevanza (allarmi rilevanti / voli con allarmi)
agg_allarmi["tasso_rilevanza"] = safe_div(
    agg_allarmi["allarmi_rilevanti"],
    agg_allarmi["voli_con_allarmi"]
).clip(0, 1)

# Fill NaN nelle colonne pct_ (rotte senza MOTIVO_ALLARME registrato)
pct_cols = [c for c in agg_allarmi.columns if c.startswith("pct_")]
agg_allarmi[pct_cols] = agg_allarmi[pct_cols].fillna(0)

print(f"✓ agg_allarmi: {agg_allarmi.shape[0]} rotte × {agg_allarmi.shape[1]} colonne")
print(f"  Null residui: {agg_allarmi.isnull().sum().sum()}")
neg_check = {c: int((agg_allarmi[c]<0).sum()) for c in _num_cols if (agg_allarmi[c]<0).sum()>0}
print(f"  Negativi dopo clip: {neg_check if neg_check else '✓ nessuno'}  ← DEVE essere nessuno")


✓ agg_allarmi: 368 rotte × 43 colonne
  Null residui: 0
  Negativi dopo clip: ✓ nessuno  ← DEVE essere nessuno


## 5. Aggregazione VIAGGIATORI per ROTTA

### 5.1 Aggregazione base

In [79]:
agg_viaggiatori = viaggiatori.groupby("ROTTA").agg(
    tot_entrati              = ("ENTRATI",           "sum"),
    tot_allarmati            = ("ALLARMATI",          "sum"),
    tot_investigati          = ("INVESTIGATI",        "sum"),
    n_osservazioni_viag      = ("ENTRATI",            "count"),
    tasso_allarme_medio      = ("tasso_allarme",      "mean"),
    tasso_inv_medio          = ("tasso_investigati",  "mean"),
    genere_predominante      = ("GENERE",             safe_mode),
    fascia_eta_predominante  = ("FASCIA_ETA",         safe_mode),
    tipo_doc_prevalente      = ("TIPO_DOCUMENTO",     safe_mode),
    nazionalita_top          = ("NAZIONALITA",        safe_mode),
    compagnia_top            = ("COMPAGNIA_AEREA",    safe_mode),
).reset_index()

# ── FIX QUALITÀ: clip negativi e cap tassi a [0,1] dopo aggregazione ────────
# Difensivo: preprocessing gestisce già i negativi a livello riga,
# ma la media aggregata può superare [0,1] per outlier residui
agg_viaggiatori["tot_entrati"]         = agg_viaggiatori["tot_entrati"].clip(lower=0)
agg_viaggiatori["tot_allarmati"]       = agg_viaggiatori["tot_allarmati"].clip(lower=0)
agg_viaggiatori["tot_investigati"]     = agg_viaggiatori["tot_investigati"].clip(lower=0)
agg_viaggiatori["tasso_allarme_medio"] = agg_viaggiatori["tasso_allarme_medio"].clip(0, 1)
agg_viaggiatori["tasso_inv_medio"]     = agg_viaggiatori["tasso_inv_medio"].clip(0, 1)

print(f"✓ agg_viaggiatori: {agg_viaggiatori.shape[0]} rotte × {agg_viaggiatori.shape[1]} colonne")
print(f"  tasso_inv_medio: min={agg_viaggiatori['tasso_inv_medio'].min():.4f}, max={agg_viaggiatori['tasso_inv_medio'].max():.4f}")
print(f"  tot_entrati:     min={agg_viaggiatori['tot_entrati'].min():.0f}, max={agg_viaggiatori['tot_entrati'].max():.0f}")

✓ agg_viaggiatori: 467 rotte × 12 colonne
  tasso_inv_medio: min=0.0000, max=1.0000
  tot_entrati:     min=0, max=30361


### 5.2 Pivot ESITO_CONTROLLO + score di rischio

Contiamo quante volte ogni esito appare per rotta.  
**Respinto** (azione di sicurezza eseguita) e **Fermato** sono i segnali più forti.

In [80]:
esiti_pivot = (
    viaggiatori.pivot_table(
        index      = "ROTTA",
        columns    = "ESITO_CONTROLLO",
        values     = "ENTRATI",
        aggfunc    = "count",
        fill_value = 0
    )
    .reset_index()
)
esiti_pivot.columns.name = None
esiti_pivot = esiti_pivot.rename(columns={
    "SEGNALATO" : "n_segnalati",
    "IN ATTESA"  : "n_in_attesa",
    "RESPINTO"   : "n_respinti",
    "FERMATO"    : "n_fermati",
    "OK"         : "n_ok",
})

agg_viaggiatori = agg_viaggiatori.merge(esiti_pivot, on="ROTTA", how="left")
for col in ["n_segnalati", "n_in_attesa", "n_respinti", "n_fermati", "n_ok"]:
    if col in agg_viaggiatori.columns:
        agg_viaggiatori[col] = agg_viaggiatori[col].fillna(0).astype(int)

# Tassi di rischio esiti (normalizzati su n_osservazioni)
agg_viaggiatori["tasso_respinti"] = safe_div(
    agg_viaggiatori["n_respinti"], agg_viaggiatori["n_osservazioni_viag"]
)
agg_viaggiatori["tasso_fermati"] = safe_div(
    agg_viaggiatori["n_fermati"], agg_viaggiatori["n_osservazioni_viag"]
)

# Score rischio esiti: respinto pesa 60%, fermato 40%
agg_viaggiatori["score_rischio_esiti"] = (
    agg_viaggiatori["tasso_respinti"] * 0.6 +
    agg_viaggiatori["tasso_fermati"]  * 0.4
).clip(0, 1)

print(f"✓ agg_viaggiatori con esiti: {agg_viaggiatori.shape[0]} rotte × {agg_viaggiatori.shape[1]} colonne")
print(f"  score_rischio_esiti: min={agg_viaggiatori['score_rischio_esiti'].min():.4f}, max={agg_viaggiatori['score_rischio_esiti'].max():.4f}")
print(f"  Null residui: {agg_viaggiatori.isnull().sum().sum()}")

✓ agg_viaggiatori con esiti: 467 rotte × 20 colonne
  score_rischio_esiti: min=0.0000, max=0.6000
  Null residui: 0


## 6. Outer Join: ALLARMI ∪ VIAGGIATORI

Usiamo un **outer join** per mantenere tutte le 567 rotte distinte,
incluse quelle presenti in un solo dataset.

In [81]:
features = agg_allarmi.merge(agg_viaggiatori, on="ROTTA", how="outer")

print(f"Shape dopo outer join: {features.shape}")
print(f"\nCopertura rotte:")
both  = (features["n_osservazioni_allarmi"] > 0) & (features["n_osservazioni_viag"] > 0)
only_a = (features["n_osservazioni_allarmi"] > 0) & features["n_osservazioni_viag"].isna()
only_v = features["n_osservazioni_allarmi"].isna() & (features["n_osservazioni_viag"] > 0)
print(f"  Entrambi i dataset:  {both.sum()}")
print(f"  Solo ALLARMI:        {only_a.sum()}")
print(f"  Solo VIAGGIATORI:    {only_v.sum()}")

Shape dopo outer join: (567, 62)

Copertura rotte:
  Entrambi i dataset:  268
  Solo ALLARMI:        98
  Solo VIAGGIATORI:    199


### 6.1 Gestione null dopo outer join

- `ZONA` / `PAESE_PART`: null per le 201 rotte solo in VIAGGIATORI → fill **"ND"**
- Colonne numeriche: null dove la rotta manca in un dataset → fill **0**
- Colonne categoriche: null residui → fill **"ND"**

In [82]:
# ── FIX PAESE_PART/ZONA: usa viaggiatori per rotte senza allarmi ──────────────
# 200 rotte sono solo in VIAGGIATORI → PAESE_PART sarebbe ND senza questo fix
paese_viag = (
    viaggiatori.groupby("ROTTA")
    .agg(
        PAESE_PART_viag = ("PAESE_PART", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "ND"),
        ZONA_viag       = ("ZONA",       lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "ND"),
    )
    .reset_index()
)
features = features.merge(paese_viag, on="ROTTA", how="left")

# Priorità: allarmi → viaggiatori → "ND"
features["PAESE_PART"] = (
    features["PAESE_PART"]
    .replace(["ND", "//", ""], None)
    .fillna(features["PAESE_PART_viag"])
    .fillna("ND")
)
features["ZONA"] = (
    features["ZONA"]
    .replace(["ND", "//", ""], None)
    .fillna(features["ZONA_viag"])
    .fillna("ND")
)
features = features.drop(columns=["PAESE_PART_viag", "ZONA_viag"])

# Colonne numeriche: fill con 0
num_cols = features.select_dtypes(include="number").columns
features[num_cols] = features[num_cols].fillna(0)

# Colonne categoriche restanti: fill con "ND"
cat_cols = features.select_dtypes(include="object").columns.drop("ROTTA")
for col in cat_cols:
    features[col] = features[col].fillna("ND")

null_total = features.isnull().sum().sum()
nd_paese   = (features["PAESE_PART"] == "ND").sum()
neg_cols   = {c: int((features[c] < 0).sum()) for c in features.select_dtypes("number").columns if (features[c] < 0).any()}

print(f"Null totali dopo fillna: {null_total}")
print(f"Rotte ancora ND dopo fix: {nd_paese}  (erano 200 prima del fix)")
print(f"Valori negativi: {neg_cols if neg_cols else 'nessuno'}")


✓ Null totali dopo fillna: 0
✓ Valori negativi:         0
✓ Shape finale:            (567, 62)


## 7. Score Composito di Rischio

Combina in modo pesato le feature più significative in un unico indice [0, 1]:

| Componente | Peso | Razionale |
|---|---|---|
| `tot_allarmi_log` normalizzato | 35% | Volume totale allarmi (log per outlier) |
| `score_rischio_esiti` | 35% | Azioni di sicurezza eseguite (respinti, fermati) |
| `pct_interpol` | 15% | Frazione allarmi INTERPOL (rischio internazionale) |
| `tasso_rilevanza` | 15% | Allarmi rilevanti su voli con allarmi |

In [83]:
log_max = features["tot_allarmi_log"].max()

features["score_composito"] = (
    (features["tot_allarmi_log"] / max(log_max, 1)) * 0.35 +
    features["score_rischio_esiti"]                 * 0.35 +
    features["pct_interpol"]                        * 0.15 +
    features["tasso_rilevanza"]                     * 0.15
).clip(0, 1)

print("score_composito:")
print(features["score_composito"].describe().to_string())
print("\n── Top 10 rotte per rischio ──────────────────────────────────────────")
top10_cols = ["ROTTA", "PAESE_PART", "score_composito",
              "tot_allarmi_log", "pct_interpol", "score_rischio_esiti", "tasso_rilevanza"]
top10 = (features[top10_cols]
         .sort_values("score_composito", ascending=False)
         .head(10))
print(top10.to_string(index=False))

score_composito:
count   567.0000
mean      0.1646
std       0.1327
min       0.0000
25%       0.0210
50%       0.1775
75%       0.2661
max       0.5634

── Top 10 rotte per rischio ──────────────────────────────────────────
  ROTTA  PAESE_PART  score_composito  tot_allarmi_log  pct_interpol  score_rischio_esiti  tasso_rilevanza
LHR-VCE Regno Unito           0.5634           7.7965        0.4359               0.5333           0.5000
CMN-BLQ     Marocco           0.5425           6.8244        0.3043               0.4000           1.0000
TUN-BLQ     Tunisia           0.5058           9.2625        0.1000               0.6000           0.0000
SIN-MXP   Singapore           0.5054           7.2703        0.0000               0.6000           0.5000
ALG-MXP     Algeria           0.4848           5.1874        0.2500               0.4000           1.0000
LGW-MXP           ?           0.4806          11.5450        0.1791               0.1893           0.2500
CMN-FCO     Marocco           0.4

## 8. Quality Report finale

In [84]:
sep = "=" * 62
print(sep)
print("  QUALITY REPORT — features_classical.csv")
print(sep)
print(f"\n  Shape: {features.shape[0]} rotte × {features.shape[1]} colonne")

# Null
nulls = features.isnull().sum()
nulls = nulls[nulls > 0]
print(f"\n  Null: {len(nulls)} colonne con null → {'✓ nessuno' if len(nulls)==0 else nulls.to_dict()}")

# Negativi
num = features.select_dtypes(include="number")
negs = {c: int((features[c] < 0).sum()) for c in num.columns if (features[c] < 0).sum() > 0}
print(f"  Negativi: {'✓ nessuno' if not negs else negs}")

# Riepilogo feature chiave
print("\n  Statistiche feature principali:")
key_feat = ["tot_allarmi_sum", "tot_allarmi_log", "pct_interpol", "pct_sdi",
            "tasso_chiusura", "tasso_rilevanza",
            "tasso_allarme_medio", "tasso_inv_medio",
            "score_rischio_esiti", "score_composito"]
print(features[key_feat].describe().T[["mean","std","min","max"]].to_string())

print(f"\n  Copertura rotte:")
print(f"    Entrambi i dataset: {((features['n_osservazioni_allarmi']>0) & (features['n_osservazioni_viag']>0)).sum()}")
print(f"    Solo ALLARMI:       {((features['n_osservazioni_allarmi']>0) & (features['n_osservazioni_viag']==0)).sum()}")
print(f"    Solo VIAGGIATORI:   {((features['n_osservazioni_allarmi']==0) & (features['n_osservazioni_viag']>0)).sum()}")
print(sep)

  QUALITY REPORT — features_classical.csv

  Shape: 567 rotte × 63 colonne

  Null: 0 colonne con null → ✓ nessuno
  Negativi: ✓ nessuno

  Statistiche feature principali:
                        mean       std    min         max
tot_allarmi_sum     739.0705 6101.4559 0.0000 103254.0000
tot_allarmi_log       3.1694    3.0002 0.0000     11.5450
pct_interpol          0.1150    0.1984 0.0000      1.0000
pct_sdi               0.1293    0.2137 0.0000      1.0000
tasso_chiusura        0.2441    0.4290 0.0000      1.0000
tasso_rilevanza       0.0685    0.2276 0.0000      1.0000
tasso_allarme_medio   0.2258    0.3201 0.0000      1.0000
tasso_inv_medio       0.7478    0.4030 0.0000      1.0000
score_rischio_esiti   0.1172    0.1656 0.0000      0.6000
score_composito       0.1646    0.1327 0.0000      0.5634

  Copertura rotte:
    Entrambi i dataset: 268
    Solo ALLARMI:       98
    Solo VIAGGIATORI:   199


## 9. Salvataggio

In [85]:
# Feature numeriche per i modelli ML
numeric_feature_cols = [
    c for c in features.select_dtypes(include="number").columns.tolist()
    if c != "score_composito"
]

meta = {
    "feature_cols" : numeric_feature_cols,
    "target"       : "score_composito",
    "n_routes"     : len(features),
    "n_features"   : len(numeric_feature_cols),
    "description"  : "Feature aggregate per ROTTA — pipeline classica"
}

out_csv  = PROC_DIR / "features_classical.csv"
out_json = PROC_DIR / "feature_cols.json"

features.to_csv(out_csv, index=False)
with open(out_json, "w") as f:
    json.dump(meta, f, indent=2)

print(f"✓ features_classical.csv  — {len(features)} rotte × {features.shape[1]} colonne")
print(f"✓ feature_cols.json       — {len(numeric_feature_cols)} feature numeriche")
print(f"\nPercorso: {out_csv}")

✓ features_classical.csv  — 567 rotte × 63 colonne
✓ feature_cols.json       — 54 feature numeriche

Percorso: /Users/danielegiovanardi/classical-vs-multiagent/classical-vs-multiagent/data/processed/features_classical.csv


---
## Riepilogo

| Step | Output | Dimensioni |
|------|--------|-----------|
| Pivot OCCORRENZE | `occ_pivot` | 368 rotte × 31 col |
| % Motivo allarme | `motivo_counts` | 351 rotte × 6 col |
| Meta ALLARMI | `meta_allarmi` | 368 rotte × 6 col |
| **agg_allarmi** | aggregazione ALLARMI completa | **368 rotte × 43 col** |
| Esiti VIAGGIATORI | `esiti_pivot` | 467 rotte × 6 col |
| **agg_viaggiatori** | aggregazione VIAGGIATORI completa | **467 rotte × 20 col** |
| **features (outer join)** | dataset ML finale | **567 rotte × 63 col** |

### Prossimo notebook
**`03_baseline_construction.ipynb`** — Costruzione della baseline storica per rotta:
media mobile, deviazione standard, soglie dinamiche per la detection delle anomalie.
